In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.signal import find_peaks
from scipy import stats
import seaborn as sns
import networkx as nx

In [ ]:
# Loading and running file
%run skel.py # executes the file

In [ ]:
# Spatial distribution of imaged neurons
# Plotting both channels
fig, ax = plt.subplots(figsize=(8, 8))
# Plot channel 1  
ax.scatter(xc[:len(xc1)], yc[:len(xc1)], s=20, c='deepskyblue', alpha=0.7,edgecolors='navy', linewidth=0.5, label=f'Channel 1 ({len(xc1)} cells)')
# Plot channel 2 
ax.scatter(xc[len(xc1):], yc[len(xc1):], s=20, c='salmon', alpha=0.7, edgecolors='darkred',linewidth=0.5,label=f'Channel 2 ({len(xc2)} cells)')
ax.set_aspect('equal')  # Equal aspect ratio (1 unit on x-axis = 1 unit on the y-axis)
ax.set_xlabel('X coordinates', fontsize=12)
ax.set_ylabel('Y coordinates', fontsize=12)
ax.set_title('Physical locations of cells in 2D spatial representation', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='best', framealpha=0.9, fontsize=7)
plt.tight_layout()
plt.show()

The diagram illustrates a two-dimensional spatial representation of neurons from the cortex of a mouse obtained through a two-photon calcium imaging experiment conducted in vivo.  The experiment uses a novel system called Diesel2p (Dual Independent Enhanced Scan Engines for Large field of view Two-Photon imaging) to image neurons expressing GCaMP6s calcium indicator (Yu et al., 2021). In this data set two adjacent strips each having an area of 5x 1.5 mm2 were imaged and the calcium signals from the first strip were characterized as channel 1 and the second strip as channel 2. The raw calcium signals were converted to a normalized measure of fluorescence change (deltaF/F). Fluorescence recording from 4255 neurons were detected of which 2167 cells were from channel 1 and 2088 cells were from channel 2. The diagram shows the physical locations of neurons from both channels plotted in 2D cartesian plane using their x-coordinates (x-axis) and y-coordinates (y-axis) of the neurons. The neurons in channel 1 are represented by blue dots whereas those in the 2nd channel are represented by pink dots. There are some dense neuronal clusters observable on the plot with overlapping neuronal regions such as in channel 1 from X-coordinates 1000 to 1500 and Y-coordinates 0 to 500, whereas there are some regions with sparse neuronal populations conspicuous as well such as in channel 1 from X-coordinates 0 to 500 and Y-coordinates 500 to 1000. Such changes are evident in channel 2 as well an example of a dense cluster would be X-coordinates 1500 to 2000 and Y-coordinates 1500 to 2000 whereas an example of a sparse cluster would be X-coordinates from 2500 to 3000 and Y-coordinates from 2500 to 3000. The overall spread of neurons is random.

In [ ]:
# Population activity heatmap (sorted by activity level)

# Peak frequency content:
from scipy import signal
activity_measure = np.zeros(dFonF.shape[0]) 
for i in range(dFonF.shape[0]):
    freqs, psd = signal.welch(dFonF[i,:], fs=fr)
    freq_mask = freqs > 0.1
    activity_measure[i] = np.sum(psd[freq_mask])  # sum power above 0.1Hz
# means sums across columns for each row (shape= 4255= number of cells)
# higher value of activity measure = more active the neuron would be
# Activity measure = 1D array of total activity per neuron

sort_indices = np.argsort(activity_measure)[::-1] # np.argsort(Activity measure) -> sorts the activity measure 1D array in ascending order (lowest activity to highest)
# [::-1] reverses for descending order (highest activity to lowest activity)

sorted_dFonF = dFonF[sort_indices, :] # sorts the dFonF matrix the 1st row of this now contains the neuron with the highest activity and so on...
sorted_activity = activity_measure[sort_indices] # sorts the activity measure from highest activity to the lowest 
sorted_xc = xc[sort_indices]# sort the xc and yc as well in accordance with the highest activity level
sorted_yc = yc[sort_indices]

# Plotting heatmap
plt.figure(figsize=(10, 4))
# Determining color scale limits - Using a percentile-based method to handle outliers
vmin = np.percentile(sorted_dFonF, 1)  # 1st percentile
vmax = np.percentile(sorted_dFonF, 99)  # 99th percentile
im = plt.imshow(sorted_dFonF, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax, extent=[t[0], t[-1], ncells-0.5, -0.5])  # Note: imshow has y inverted
# aspect= auto - no fixed aspect ratio - as neurons (y-axis) and time (x-axis) have different scales
# extent shows x axis- from t[0] to t[-1] and y-axis shows neurons index, the -0.5 offset ensures cells are centered in their rows
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('Cells (sorted by activity)', fontsize=12)
plt.title('Fluoroscence time series for all cells', fontsize=14)
cbar = plt.colorbar(im, pad=0.01) # adding colorbar
cbar.set_label(r'$\Delta$F/F', fontsize=12)
time_interval = 50  # seconds
if t[-1] > time_interval:
    plt.xticks(np.arange(0, t[-1]+time_interval, time_interval))
# place ticks at 0,50,100, ...
plt.tight_layout()
plt.show()

The plot shows the deltaF/F fluorescence time series for all 4255 cells from both channels as a matrix (Cells x Time) with the colour scale indicating signal. Neural circuits exhibit stimulus evoked temporal activity patterns and by tracking the calcium surges which are indicative of neuronal action potential, we can understand and map the underlying functional connections of a neural network (Patel et al., 2025).
The fluorescence signal corresponds to neurons or cells expressing the genetically encoded calcium indicator GCaMP6s and the strength of the signal is related to the level of intracellular Ca+2 increase. When neuron fires voltage gated calcium channels open and there is an influx of Ca+2 into the neuron, the calcium binds to GCaMP6s and the indicator fluoresces. A high deltaF/F or dFonF (indicated by more warmer colours on the plot) indicates a large increase in intracellular Ca+2 levels which is indicative of a high firing rate, synchronous activity or prolonged activity of the neuron. When there are closely spaced action potential (AP), the Ca+2 level from previous AP has not reached baseline and a new AP arrives. The residual Ca+2 remains from the previous spike and more Ca+2 enters the cell due to new AP this results in a temporal summation of intracellular Ca+2 levels and therefore an increase in deltaF/F, this is referred as Treppe or staircase effect. A lower value or negative value of deltaF/F (indicated by more cooler colours in the blue/violet region of the spectrum of the colour bar) indicates quiescence or baseline action potential in which the calcium levels in the neuron’s cytosol are at its resting level and the neuron is not firing an action potential. A negative dFonF indicates hyperpolarisation due to a strong sustained inhibitory input which can result in closure of voltage gated calcium ion channels and pumps reducing intracellular calcium ion levels below its resting level.
The time in seconds is indicated on the x-axis whereas the neurones are present on y-axis and they are sorted with respect to their activity level with the most active neurons at the top of the graph. The amount of activity for each cell is measured using peak frequency content which is sensitive to temporal network structures and helps identify neuronal synchronization and connectivity patterns. Peak frequency measures the dominant oscillatory component in a neural signal. This method is preferred instead of integrated activity measure as that considers absolute value of dFonF so would also include activity in negative direction that is hyperpolarization of neurons. Variance measure was not used as it is not robust to noise and mean firing rate requires spike count obtained through spike deconvolution which is an error-prone procedure to obtain.
In the plot there is a wider horizontal band of sustained activity from t=690s to t-710s across many neurons which is suggestive of a tonically active neuron firing continuously (treppe effect). Moreover, several vertical yellow lines are visible such as around t=290s, t=330s and t=360s which indicates a population event in which many neurons are co-activated synchronously by a strong sensory stimulus. Synchronisation is a hallmark of interconnected dynamic systems and it is critical in binding neurons into functional ensembles. This temporal alignment allows neuronal assemblies to process information with increased efficiency (Patel et al., 2015).

In [ ]:
# Calcium transients in highly active cells

all_traces = dFonF.flatten()
median_val = np.median(all_traces)
mad = 1.4826 * np.median(np.abs(all_traces - median_val))
min_height = median_val + 4 * mad  # 4× above median 
min_distance_frames = int(fr * 0.5)  # minimum frames between peaks
n_transients_threshold = 20 # # Minimum number of transients to be considered an active cell
candidate_cells = [] # initialise empty list to store information about qualifying neurons
for cell_idx in range(ncells): # ncells= total number of neurons -> iterate through these
    trace = dFonF[cell_idx, :] # extract all the time points from dFonF for a particular cell
    peak_indices, properties = find_peaks(trace, height=min_height, distance=min_distance_frames,prominence=min_height/2)
    if len(peak_indices) >= n_transients_threshold: 
        peak_heights = trace[peak_indices] 
        mean_height = np.mean(peak_heights)
        std_height = np.std(peak_heights)
        if mean_height > 3 and std_height/mean_height < 1.0:  # Reasonable sized + consistent transients 
            candidate_cells.append({'cell_idx': cell_idx,'n_peaks': len(peak_indices),'mean_height': mean_height,'peak_indices': peak_indices,'trace': trace})

candidate_cells.sort(key=lambda x: x['n_peaks'], reverse=True) # Sorting neurons with most active neurons first
if len(candidate_cells) >= 5: # check if candidate cell list at least has 5 elements
    selected_cells = candidate_cells[:5] # Select 5 cells out of candidate cells
    
# Plotting
fig, axes = plt.subplots(5, 1, figsize=(15, 10), sharex=True) # all subplots share the same x axis (sharex=True)
vertical_spacing = 2.0  # Preventing overalpping: vertical spacing between traces
for i, (cell_info, ax) in enumerate(zip(selected_cells, axes)): # Plot each cell's trace
    cell_idx = cell_info['cell_idx'] # Extracting cell information (index, DFonF fluoroscent trace, peak locations)
    trace = cell_info['trace']
    peak_indices = cell_info['peak_indices']
    offset = i * vertical_spacing # Apply vertical offset 
    shifted_trace = trace + offset
    ax.plot(t, shifted_trace, linewidth=1.5, alpha=0.8, label=f'Cell {cell_idx} ({len(peak_indices)} transients)') # Plotting the traces
    ax.axhline(y=offset, color='gray', linestyle='--', alpha=0.5, linewidth=0.8) # Dashed gray line at baseline 
    ax.text(t[-1] + 2, offset + 0.1, f'Cell {cell_idx}',verticalalignment='center', fontweight='bold', fontsize=10) # Labels which neuron the trace belongs to
    ax.set_yticks([offset]) # y-axis ticks - add label 0 dFonF to baseline neuron
    ax.set_yticklabels(['0 ΔF/F'])
    ax.grid(True, alpha=0.3, axis='x') 
    ax.spines['top'].set_visible(False) # Removes borders 
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
axes[-1].set_xlabel('Time (s)', fontsize=12) # Set Common x-axis label
fig.suptitle('Calcium Transients in 5 Active Cells', fontsize=16, fontweight='bold', y=0.95)
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust Layout (leave space at top for title)
plt.show()

The diagram shows a plot of the fluorescence time series for 5 cells with calcium transients. The 5 traces of cells are stacked vertically on a shared x-axis. A calcium transient is a surge in intracellular calcium ion concentration in a neuron triggered by one or more action potentials. The calcium transients are first shortlisted by identifying the peaks in the data and the peaks are separated by regions of low activity that is referred to as the minimum distance between peaks. This is followed by selecting traces with at least 20 transients present, have a reasonable size and are consistent. The cells are sorted according to their activity level and five cells with the highest activity are selected and plotted. The peaks on the graph indicate Action Potential or Calcium transients and the baseline activity level (resting intracellular Calcium level) 0 deltaF/F is also indicated by the gray line on the graph. The activity is similar in the 3 upper neurons (Cell 4202, 3649,1410) initially during the time window 0 to 200seconds with a high firing rate, large number of peaks and larger amplitude of peaks indicating bursts of activity. The firing of cell 48 during this interval is really low whereas the activity of Cell 1965 is intermediary, with spikes being visible but the amplitude of increase in dFonF is lower in comparison to the cells above (4202,3649,1410). The intermediary level indicates Action Potential arriving at the neuronal surface resulting in an increase in intracellular Calcium levels but the EPSPs (Excitatory Post Synaptic Potential) are not too closely spaced and not strong enough to result in a massive surge in Calcium levels but they still result in a reasonable increase in intracellular Calcium levels which is detected and the dFonF value is in between the silent and bursting neuronal states. From 200 to 300 seconds the activity is lower in cell 3649,1410 intermediary in cell 4202 and high in cell 1965 and cell48.  The activity remains intermediary in cell 4202 and low to intermediary in cell 1965, for the remaining time interval. The activity of Cell 3649 and 48 also remains intermediary with an increase at the very end from around t=600s to t=700 seconds. The activity cell 1410 remains low for most of the remaining interval apart from a surge in dFonF detected from t=380s to t=420s.

In [ ]:
# Distribution of pairwise correlations

corr_matrix = np.corrcoef(dFonF1)  # Calculating correlation matrix - Shape: (ncells x ncells)
upper_tri = np.triu_indices_from(corr_matrix, k=1) # Extracting upper triangle - exclude repaeated values
corr_values = corr_matrix[upper_tri]

mean_corr = np.mean(corr_values)
median_corr = np.median(corr_values)
std_corr = np.std(corr_values)
skew = stats.skew(corr_values)
kurtosis = stats.kurtosis(corr_values)
n_bins = int(np.ceil(np.log2(len(corr_values)) + 1))

# Plotting distribution of correlation coefficients
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.4)
plt.figure(figsize=(8, 4))
plt.hist(corr_values, bins=n_bins, density=True, alpha=0.75, color='#2E86AB', edgecolor='white', linewidth=1.5)
kde = stats.gaussian_kde(corr_values) # KDE smooth PDF of correlation values
x_range = np.linspace(corr_values.min(), corr_values.max(), 1000)
plt.plot(x_range, kde(x_range), 'k-', linewidth=1.5, alpha=0.8, label='Kernal Density Estimate') 
plt.axvline(mean_corr, color='#A23B72', linestyle='--', linewidth=1.0, label=f'Mean = {mean_corr:.3f}')
plt.axvline(median_corr, color='#F18F01', linestyle='--', linewidth=1.0,label=f'Median = {median_corr:.3f}')
plt.axvline(0, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
stats_text = (f'Mean ± SD = {mean_corr:.3f} ± {std_corr:.3f}\n'f'Median = {median_corr:.3f}\n'f'Skewness = {skew:.3f}\n'f'Kurtosis = {kurtosis:.3f}\n'f'Range = [{corr_values.min():.3f}, {corr_values.max():.3f}]')
plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes,fontsize=9, verticalalignment='top',bbox=dict(boxstyle='round', facecolor='white', alpha=0.9,edgecolor='gray', linewidth=1))
plt.xlabel('Pearson correlation coefficient (r)', fontsize=14, fontweight='bold')
plt.ylabel('Probability density', fontsize=14, fontweight='bold')
plt.title('Distribution of pairwise neuronal correlations', fontsize=16, fontweight='bold', pad=15)
plt.legend(loc='upper right', framealpha=0.9, fontsize=11)
plt.xlim([max(corr_values.min(), -1), min(corr_values.max(), 1)])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The plot shows a histogram of the distribution of Pearson correlation coefficients (r) between the neuronal pairs. Most correlations occur when r=zero indicating independent neural activity and really limited functional connectivity between the neuronal pairs which is common in resting state of neurons when the neurons are not actively firing there might be some fluctuating dFonF around the resting level indicating subthreshold Action Potentials arising as a cumulative result of EPSP (Excitatory Post Synaptic Potential) and IPSP (Inhibitory Post Synaptic Potential) arriving at the neuronal surface. The graph shows a unimodal distribution which indicates similar cell populations and functional modules. The distribution has a skewness of 1.075 and kurtosis of 2.949, as the approximate skewness and kurtosis are close to 1 and 3 respectively the plot follows a fairly normal distribution (indicated by the kernel density estimate as well). The graph shows some higher positive correlation around 0.1 which indicates synchronized population activities with multiple neurons receiving input from the same presynaptic neuron creating a functional cell assembly by triggering different parts of the brain to produce a specific coordinated response to the stimulus. There might be reciprocal connections present between neurons which have feedback loops present that amplify the correlations present between neurons. Also, monosynaptic connections tend to create stronger correlations. The negative correlations might be a result of direct inhibition that is when one neuron is excited it causes a feedback inhibition of the second neuron. The negative correlation might be a result of neuronal firing being out of phase. The opposite activity between neuronal pairs prevents network saturation with all neurons active around the same time and avoids information overload. In the plot of channel 1 neuronal population mean and median are 0.017 and 0.011 respectively which indicates mostly sparse connectivity of the neuronal network with only a  few strong connections.

In [ ]:
#  Adjacency matrix

# dFonF1 shape: (ncells, nframes)
corr_matrix = np.corrcoef(dFonF1)  # Calculates pearson correlation coefficients between neuronal pairs - Shape: (ncells, ncells)
np.fill_diagonal(corr_matrix, np.nan) # Diagonals are always 1 - remove them (make them NaN)
all_corrs = corr_matrix[~np.isnan(corr_matrix)].flatten() #~np.isnan - selects all nonNan values, flatten the array
ncells1 = np.size(dFonF1,0)
x = 96  # threshold value (choose this parameter)
threshold = np.percentile(all_corrs, x) # means if threshold is 0.75 than 98% of correlations are <0.75
print(f"Threshold at {x}th percentile: {threshold:.4f}")

adj_matrix = (corr_matrix > threshold).astype(int) # Binary adjacency matrix,  1=strong connection (correlation> threshold) and 0 otherwise
np.fill_diagonal(adj_matrix, 0) # diagonals = 0 (no self-edges)
adj_matrix = np.maximum(adj_matrix, adj_matrix.T)  # Ensure symmetry (undirected graph) # this is added new line test if it works property

num_edges = np.sum(adj_matrix)//2 # Matrix is symmetric so edges are counted twice - for undirected graphs number of unique edges is obtained by dividing by 2
max_edges = ncells1 * (ncells1 - 1) // 2 
expt_edges= max_edges * ((100-x)/100) 
sparsity = 100 * (1 - num_edges / max_edges)
density = 100 - sparsity

print(f"Number of neurons: {ncells1}")
print(f"Number of edges: {num_edges}")
print(f"Expected number of edges: {expt_edges}")
print(f"Edge sparsity: {sparsity:.2f}%")
print(f"Connection density: {density:.2f}%")

# Plotting the adjacency matrix
plt.figure(figsize=(10, 8))
plt.imshow(adj_matrix, cmap='binary', interpolation='none', aspect='equal', vmin=0, vmax=1)
plt.colorbar(label='Edge (0 = No Edge, 1 = edge present)')
plt.title(f'Adjacency Matrix (Threshold: {x}th percentile, {num_edges} edges)')
plt.xlabel('Neuron ID')
plt.ylabel('Neuron ID')
plt.text(0.02, 0.98, f'Neurons: {ncells1}\nEdges: {num_edges}\nSparsity: {sparsity:.1f}%',transform=plt.gca().transAxes, fontsize=10,verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

A symmetric undirected network is created where nodes represent neurons and edges indicate a binary variable which is one if there is a connection present between the neuronal pair and zero if it is absent (not weighted). The binary adjacency matrix is created by choosing a threshold value which is 96th percentile of the correlation coefficient distribution and has a value of 0.1021 i.e., if the Pearson correlation coefficient value between 2 cells is greater than 0.1021 this is considered a strong meaningful interaction between the two neurons to indicate a physical connection between the neuronal pair otherwise, we consider the 2 neurons are not connected.  The threshold is chosen to be sufficiently high to reduce the number of edges and increase the computational efficiency however it is still sufficiently low enough to observe the underlying network architecture. The edge sparsity or fraction of missing connections is 96% and he connection density, the fraction of all existing connections in the underlying network is 4%. The number of neurons in channel 1 is 2167 with 93874 total undirected edges between the nodes at 96% edge sparsity. The binary adjacency matrix is constructed and plotted with neuronal index indicated on the x and y axis and a color bar indicating the presence or absence of a connection. A black dot (one) indicates edge present between two nodes whereas white dot (zero) shows the absence of a connection). The graph is symmetric with the matrix being mirrored along the diagonal. The diagonal contains all white pixels indicating no self-connections. The plot indicates certain columns with white bands which indicate isolated neurons that do not have a strong correlation to any other neurons in the network these occur from neuron index 230 to 250, from 750 to 780 and from 1000 to 1050. There are some denser black bands across some rows although they are not continuous indicating that those particular neurons form connections with many other neurons but not all of the neurons present in the network, these occur from neuron index 700 to 750 and from 1100 to 1250. Overall, there is a random scattering of connections with some densely connected neurons forming hubs and others more sparsely connected to only a few other neurons. Humphries (2017) highlight the functional network of neuronal activity derived from the fluorescence recordings does not directly map onto the underlying anatomical synaptic wiring. Consequently, correlated activity between neurons should not be interpreted as evidence of direct causal interaction or anatomical connectivity. The functional interactions between neurons can change rapidly without any physical rewiring due to integration of multiple inputs and short-term plasticity (Humphries, 2017). However, prolonged activity can serve as evidence to changing physical structure of the network (Baeg et al., 2007).

In [ ]:
# Visualising network with real spatial coordinates
# Creating graph from adjacency matrix
G = nx.from_numpy_array(adj_matrix)
# Positioning nodes according to x and y coordinates
pos = {i: (xc1[i], yc1[i]) for i in range(ncells1)}
# Plotting
plt.figure(figsize=(6, 8))
nx.draw_networkx_nodes(G, pos, node_color='#1f77b4', node_size=10, alpha=0.8, edgecolors='black') #drawing nodes
nx.draw_networkx_edges(G, pos, alpha=0.1, edge_color='gray') # drawing edges
plt.title('Network Structure for Channel 1', fontsize=14)
plt.axis('off')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

Constructing a multi-neuron network model comprises of two key elements, the nodes which are the neurons and the edges that are the interactions or connectivity between neuronal pairs (Humphries, 2017). The graph shows the physical location of the nodes (neurons) plotted using the x and y coordinates of neurons in channel 1 and the gray lines indicate edges in between 2 neurons. Some hubs or neurons that are densely connected to many other neurons are observable such as towards the extreme left and center of the plot. There is a really dense connectivity between neurons at the bottom of the plot and also at the top right indicative of synchronization and creation of functional assemblies of neurons which are co-activated.

In [ ]:
# Network summary statistics
# (a) Calculating Average degree
degrees = np.sum(adj_matrix, axis=1)  # Degree of each node
average_degree = np.mean(degrees)
print(f"\nAverage degree ⟨k⟩ = {average_degree:.2f}")

# (b) Calculating Average clustering coefficient
clustering_local = nx.clustering(G) # Local clustering coefficient for each node
average_clustering = np.mean(list(clustering_local.values()))
print(f"\nAverage clustering coefficient = {average_clustering:.4f}")

# (c) Calculating Global clustering coefficient 
global_clustering = nx.transitivity(G)
print(f"Global clustering coefficient (transitivity) = {global_clustering:.4f}")

The average degree indicates the average number of connections per neuron for the overall network. In the neural network under consideration the average degree = 86.64 which indicates that each neuron is approximately connected to 87 other neurons out of the 2166 (excluding self-edges) total available. This indicates extensive local interactions but it is still sparse overall as the connection density is only 4%. The average clustering coefficient is the average of the local clustering coefficient and it is the average probability that two adjacent nodes of a neuron are also connected to each other. For the given data the average clustering coefficient is 0.5146 indicting a strong local network i.e., there is a51.5% chance that any two adjacent nodes are connected. This suggests presence of functional assemblies and indicates pronounced triadic closures and reciprocal neuronal connections. The global clustering coefficient measures the fraction of closed triangles or connected triplets in the network. The global clustering is 0.63 which is higher than the average clustering as well and indicates a heterogenous clustering in the network and presence of hubs with really high clustering which increase the global average. The network exhibits small world properties that is high local clustering and short path lengths.

In [ ]:
# Degree distribution vs. Erdos Renyi 
# Plottig
fig, axes = plt.subplots(3, 1, figsize=(6, 10))

# degree distribution 
degrees = [deg for _, deg in G.degree()]
mean_degree = np.mean(degrees)
hist_counts, bins = np.histogram(degrees, bins=50)
bin_centers = 0.5 * (bins[1:] + bins[:-1])
Pk = hist_counts / len(degrees)
axes[0].bar(bin_centers, Pk, width=bins[1]-bins[0], alpha=0.7, color='skyblue', edgecolor='black')
axes[0].axvline(mean_degree, color='red', linestyle='--', linewidth=2, label=f'Mean degree = {mean_degree:.2f}')
axes[0].set_xlabel('Degree (k)')
axes[0].set_ylabel('P(k)')
axes[0].set_title(f'Degree Distribution (n={ncells} neurons)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Log-log plot - to check for power law (scale-free)
degree_values, degree_counts = np.unique(degrees, return_counts=True)
Pk = degree_counts / len(degrees)
non_zero_mask = (Pk > 0) & (degree_values > 0) # Remove zeros for log-log plot
log_degrees = np.log10(degree_values[non_zero_mask])
log_Pk = np.log10(Pk[non_zero_mask])
axes[1].scatter(log_degrees, log_Pk, alpha=0.7, color='coral')
if len(log_degrees) > 1: # Fit linear regression to check for power law
    slope, intercept, r_value, p_value, std_err = stats.linregress(log_degrees, log_Pk)
    axes[1].plot(log_degrees, slope*log_degrees + intercept, 'r--', label=f'Slope = {slope:.2f}\nR² = {r_value**2:.2f}')
axes[1].set_xlabel(r'$\log_{10}(k)$')
axes[1].set_ylabel(r'$\log_{10}(P(k))$')
axes[1].set_title('Log-Log Degree Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Erdös-Renyi
n_edges = G.number_of_edges() # Creating Erdös-Renyi graph with same number of nodes and edges
p_er = (2 * n_edges) / (ncells * (ncells - 1))  # Connection probability
G_er = nx.erdos_renyi_graph(ncells, p_er)
degrees_er = [deg for _, deg in G_er.degree()]

hist_neural, bins = np.histogram(degrees, bins=30, range=(0, max(degrees)+1))
hist_er, _ = np.histogram(degrees_er, bins=bins)
Pk_neural = hist_neural / len(degrees)
Pk_er = hist_er / len(degrees_er)
bin_centers = 0.5 * (bins[1:] + bins[:-1])
bin_width = bins[1] - bins[0]
axes[2].bar(bin_centers - bin_width/4, Pk_neural, width=bin_width/2,alpha=0.7, color='skyblue', edgecolor='black',label=f'Neural (µ={np.mean(degrees):.1f})')
axes[2].bar(bin_centers + bin_width/4, Pk_er, width=bin_width/2,alpha=0.7, color='salmon', edgecolor='black',label=f'ER (µ={np.mean(degrees_er):.1f})')
axes[2].set_xlabel('Degree (K)')
axes[2].set_ylabel('P(k)')
axes[2].set_title('Comparison with Erdös-Renyi Network')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The top plot shows the degree distribution of the network with p(k) on the y-axis and degree(k) on the x-axis. It indicates the probability of finding connections of a particular degree in a neuronal network. An exponential decay is evident with the greatest probability (0.28) of finding around 0 connections. The mean degree is 86.64 as previously discussed in Q5 and is indicated by the red dotted line. The exponential decay is indicative of a scale free network as well with most neurons having minimal connectivity and sum neurone with really high degree. The maximum degree is around 360 connections in the network.
The network exhibits scale free topology as evidenced by the linear relationship on the log-log degree distribution however with R squared value =0.66 which indicates that 66% of the variation in y [log10(p(k))] is explained by x  [log10(k)]. This indicates that most nodes in the data have very few connections while a few nodes have an extraordinarily large number of connections and these are called hubs.  Scale- free networks emerge as a result of preferential attachment (Barbasi Albert Model) that is new nodes prefer to connect with nodes that already have many connections.

In [ ]:
# Mutual information network
ncells_total = dFonF1.shape[0] # obtain the total number of cells from the first dimension
nframes = dFonF1.shape[1] # obtain the number of time frames from 2nd dimension (columns)

# Selecting 100 most active neurons 
mean_activity = np.mean(dFonF1, axis=1) # mean activity of each neuron across all time frames
most_active_indices = np.argsort(mean_activity)[-100:]  # Sorting the mean activity in ascending order and taking the 100 most active ones
most_active_indices = np.sort(most_active_indices)  # Sorts the 100 most active neurons by their indices not their activity level
active_data = dFonF1[most_active_indices, :] # array containing data for only the 100 most active neurons
active_xc = xc1[most_active_indices] # x-coordinates of selected neurons
active_yc = yc1[most_active_indices] # y-coordinates of selected neurons

# Discretizing
epoch_duration = int(fr)  # fr=framerate=7.6Hz, 1 second = 7 frames at 7.6 Hz
n_epochs = nframes // epoch_duration
# Creating discretized matrix 
discretized = np.zeros((100, n_epochs)) # initialise array of zeros with shape (cells x n_epochs)
for epoch in range(n_epochs):
    start_frame = epoch * epoch_duration # calculates starting frame index for current epoch (like epoch1=frames7)
    end_frame = start_frame + epoch_duration # calculates ending frame index for current epoch (like epoch1=frames14)
    discretized[:, epoch] = np.mean(active_data[:, start_frame:end_frame], axis=1) # Takes data for each epoch, calculate mean, store in matrix

# Plotting discretized matrix
vmin = np.percentile(discretized, 1)  # 1st percentile
vmax = np.percentile(discretized, 99)  # 99th percentile (using perecentile instead of minimum to make visualisation robust of outliers)
plt.figure(figsize=(10, 4))
plt.imshow(discretized, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax, extent=[0, n_epochs, 0, 100]) # extent=[xmin, xmax, ymin, ymax]
plt.colorbar(label=r"Mean $\Delta F/F'$")
plt.xlabel('Time (1-second epochs)')
plt.ylabel('Neurons')
plt.title('$\Delta F/F$ temporal patterns in neurons with elevated baseline activity')
plt.tight_layout()
plt.show()

# Calculating Shannon mutual information using plugin estimator
# discretize activity into bins for entropy calculation
# Freedman-Diaconis rule for bin width
n_bins = int(np.ceil((np.max(discretized) - np.min(discretized)) / (2 * stats.iqr(discretized.flatten()) / (len(discretized.flatten())**(1/3)))))
n_bins = min(max(n_bins, 10), 50) # Bin number is between 10 and 50 (suitable for computation)
# Pairwise mutual information matrix
MI_matrix = np.zeros((100, 100)) # initialise a 100x100 matrix of zeros
for i in range(100):
    for j in range(i+1, 100):  # computes pair when j>i (upper triangle)
        hist_2d, _, _ = np.histogram2d(discretized[i, :], discretized[j, :], bins=n_bins) # creating histogram of joint activity
        p_xy = hist_2d / np.sum(hist_2d) # converting histogram count to probability distribution
        p_x = np.sum(p_xy, axis=1) # Marginal probabilities
        p_y = np.sum(p_xy, axis=0) # Marginal probabilities
        mi = 0 #initialise mutual information accumlator
        for idx_x in range(n_bins):
            for idx_y in range(n_bins):
                if p_xy[idx_x, idx_y] > 0 and p_x[idx_x] > 0 and p_y[idx_y] > 0: # include terms where joint probability>0
                    mi += p_xy[idx_x, idx_y] * np.log2(p_xy[idx_x, idx_y] / (p_x[idx_x] * p_y[idx_y])) # Apply mutual information formula
        MI_matrix[i, j] = mi # upper traingle (Store calculated value in initialised matrix)
        MI_matrix[j, i] = mi # Lowe triangle (same as it is a symmetric matrix)

# Plotting Mutual Information Distribution
plt.figure(figsize=(10, 6))
mi_values = MI_matrix[np.triu_indices(100, k=1)] # extract the upper triangle values excluding the diagonal
plt.hist(mi_values, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Mutual Information (bits)')
plt.ylabel('Frequency')
plt.title('Distribution of Pairwise Mutual Information Values')
plt.grid(True, alpha=0.3)
plt.show()

#Plotting adjacency matrix 
threshold = np.percentile(mi_values, 90) # top 10% of most strongly connected pairs from mi distribution are considered connected
adjacency = (MI_matrix > threshold).astype(float) # binary matrix 
np.fill_diagonal(adjacency, 0)  # diagonal=0 (no self connections)
plt.figure(figsize=(8, 6))
plt.imshow(adjacency, cmap='binary', interpolation='none') 
plt.colorbar(label='Connection') # black = 1(connected), white= 0(not connected)
plt.xlabel('Neuron ID')
plt.ylabel('Neuron ID')
plt.title(f'Adjacency Matrix (Threshold = {threshold:.4f} bits)')
plt.show()

# Visualising graph using spatial coordinates
G = nx.Graph() # Creating an empty undirected graph using NetworkX library
for i in range(100):
    G.add_node(i, pos=(active_xc[i], active_yc[i])) # adds position of neurons(nodes)
for i in range(100):
    for j in range(i+1, 100): # consider upper triangle
        if adjacency[i, j] > 0: #check connection=1
            G.add_edge(i, j) # add undirected edge between node i and j
pos = nx.get_node_attributes(G, 'pos') #creates a dictionary of x and y coordinates of each nide
plt.figure(figsize=(10, 10))
nx.draw_networkx_nodes(G, pos, node_size=20, node_color='lightblue', edgecolors='black', linewidths=0.5) # draw nodes
nx.draw_networkx_edges(G, pos, alpha=0.3, width=0.5) # draw edges between nodes
plt.title('Network Structure of Active Neuronal Population')
plt.axis('equal')
plt.tight_layout()
plt.show()

print(f"Total neurons analyzed: 100")
print(f"Time epochs: {n_epochs} (1-second each)")
print(f"Mutual Information - Mean: {np.mean(mi_values):.4f}, Std: {np.std(mi_values):.4f}")
print(f"Network connections: {int(np.sum(adjacency)/2)} edges")
print(f"Connection density: {100 * np.sum(adjacency)/(100*99):.2f}%")

In [ ]:
# Correcting for finite sample bias in mutual information
N = n_epochs  # number of samples
R = n_bins    # number of bins or states 
occupancy_matrices = []  # List to store occupancy matrices for each neuron
for neuron_idx in range(100): # 100 neurons so iterates 100 times
    neuron_data = discretized[neuron_idx, :] # extract data for each neuron
    hist, _ = np.histogram(neuron_data, bins=n_bins) # extracts the count (how many time the neuron was in each state)
    occupancy_matrices.append(hist) # Store the count for eaxh neuron in  initialised matrix
occupancy_matrices = np.array(occupancy_matrices)  # Converting into numpy array, Shape: (100 neurons, n_bins)

# Panzeri-Treves correction
PT_bias_matrix = np.zeros((100, 100)) # initialise a 100 by 100 matrix of zeros
for i in range(100):
    for j in range(i+1, 100): 
        hist_2d, _, _ = np.histogram2d(discretized[i, :], discretized[j, :], bins=n_bins) # obtain count from histogram of joint activity
        hist_i = occupancy_matrices[i, :] # count from occupancy matrix (marginal distributions)
        hist_j = occupancy_matrices[j, :]
        # Calculating number of non-zero bins
        R_i = np.sum(hist_i > 0)  # Effective number of states for neuron i
        R_j = np.sum(hist_j > 0)  # Effective number of states for neuron j
        R_ij = np.sum(hist_2d > 0)  # Effective number of joint state 
        # Applying Panzeri-Treves bias correction formula
        bias_pt = (R_ij - R_i - R_j + 1) / (2 * N * np.log(2))
        PT_bias_matrix[i, j] = bias_pt # storing data in initialised matrix
        PT_bias_matrix[j, i] = bias_pt # symmetric

pt_bias_values = [] # initialise a list
for i in range(100):
    for j in range(i+1, 100):
        pt_bias_values.append(PT_bias_matrix[i, j]) # list of bias values corresponding to upper triangle

pt_bias_array = np.array(pt_bias_values) # converting to numpy array
mi_array = np.array(mi_values)  # converting mutual information (mi) values to array
corrected_mi_pt = mi_array - pt_bias_array # subtract estimated bias from MI values
corrected_mi_pt[corrected_mi_pt < 0] = 0 #  MI cannot be negative - negative mi might have arised from overcorrection - it is set to zero 

# Creating the symmetric bias-corrected matrix 
MI_matrix_corrected_pt = np.zeros((100, 100)) # initialise a 100 x100 matrix of zeros
pair_idx = 0 # initialising a counter
for i in range(100):
    for j in range(i+1, 100):
        MI_matrix_corrected_pt[i, j] = corrected_mi_pt[pair_idx] # assigning values to upper triangle
        MI_matrix_corrected_pt[j, i] = corrected_mi_pt[pair_idx] # assigning same values to lower triangle (as symmetric)
        pair_idx += 1 #increasing counter by 1

print(f"Mean of Mutual Information(MI): {np.mean(mi_array):.6f} bits")
print(f"Mean Panzeri-Treves bias estimate: {np.mean(pt_bias_array):.6f} bits")
print(f"  Mean bias: {(np.mean(pt_bias_array)/np.mean(mi_array)*100):.2f}% of mean MI")
print(f"Mean bias-corrected MI: {np.mean(corrected_mi_pt):.6f} bits")

The Panzeri-Treves estimator predicts how much mutual information between two neurons is overestimated due to limited data. When sample size are limited the rare joint events may not occur making neurons appear more independent than they really are.The bias is larger when more states are possible (when R_ij is large) and is smaller with more samples (when N is large). The Mean Panzeri Treves bias is subtracted from the raw mutual information to get a better estimate. About 35% of the initial calculated Mutual Information (MI) was likely due to noise because of limited amount of data. The mean Panzeri Treves bias estimate is 0.3387 bits this means the uncorrected mutual information between those two neurons is overestimated by about 0.03387 bits due to sampling limitations.

References
Baeg, E. H., Kim, Y. B., Kim, J., Ghim, J. W., Kim, J. J., & Jung, M. W. (2007). Learning-induced enduring changes in functional connectivity among prefrontal cortical neurons. Journal of Neuroscience, 27(4), 909-918. https://doi.org/10.1523/JNEUROSCI.4759-06.2007
Humphries, M. D. (2017). Dynamical networks: Finding, measuring, and tracking neural population activity using network science. Network Neuroscience, 1(4), 324-338. https://doi.org/10.1162/NETN_a_00020
Patel, T. P., Man, K., Firestein, B. L., & Meaney, D. F. (2015). Automated quantification of neuronal networks and single-cell calcium dynamics using calcium imaging. Journal of neuroscience methods, 243, 26-38. https://doi.org/10.1016/j.jneumeth.2015.01.020
Yu, C. H., Stirman, J. N., Yu, Y., Hira, R., & Smith, S. L. (2021). Diesel2p mesoscope with dual independent scan engines for flexible capture of dynamics in distributed neural circuitry. Nature communications, 12(1), 6639. https://doi.org/10.6084/m9.figshare.15163914
